In [6]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import pyautogui
import time
import math


In [7]:
model_path = '/absolute/path/to/gesture_recognizer.task'


In [ ]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7, min_tracking_confidence=0.5)
screen_width, screen_height = pyautogui.size()
print("\n air control")
prev_screen_x, prev_screen_y = 0, 0

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()
eye_closed_frames = 0


 air control


I0000 00:00:1775055708.503088       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [9]:
#gesture time control variables
click_start_time = None
click_times = []
click_cooldown = 0.5  # Cooldown time in seconds to prevent multiple clicks
scroll_mode = False
freeze_cursor = False
screenshot_cooldown = 2
last_screenshot_time = 0  # Cooldown time in seconds to prevent multiple screenshots


In [ ]:
import cv2
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Cannot open camera")
    exit()

while True: 
    ret , frame = cap.read()
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break 

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    #face
    face_results = face_mesh.process(rgb)
    
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            thumb_tip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
            index_tip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
            middle_tip = hand_landmarks.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP]
            ring_tip = hand_landmarks.landmark[mp_hands.HandLandmark.RING_FINGER_TIP]
            pinky_tip = hand_landmarks.landmark[mp_hands.HandLandmark.PINKY_TIP]

            fingers = []

            if thumb_tip.x < hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_IP].x:
                fingers.append(1)
            else:
                fingers.append(0)

            tips = [
                mp_hands.HandLandmark.INDEX_FINGER_TIP,
                mp_hands.HandLandmark.MIDDLE_FINGER_TIP,
                mp_hands.HandLandmark.RING_FINGER_TIP,
                mp_hands.HandLandmark.PINKY_TIP
            ]
            for tip in tips:
                if hand_landmarks.landmark[tip].y < hand_landmarks.landmark[tip - 2].y:
                    fingers.append(1)
                else:
                    fingers.append(0)

            # move cursor by index finger
            screen_x = int(index_tip.x * screen_width)
            screen_y = int(index_tip.y * screen_height)
            pyautogui.moveTo(screen_x, screen_y, duration=0.05)

            # distance between thumb and index finger for click detection
            dist = math.hypot(thumb_tip.x - index_tip.x, thumb_tip.y - index_tip.y)
            if dist < 0.06:
                if not freeze_cursor:
                    freeze_cursor = True
                    pyautogui.click()
                    cv2.putText(frame, 'Click', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    click_times.append(time.time())
                else:
                    if len(click_times) > 1 and click_times[-1] - click_times[-2] < click_cooldown:
                        pyautogui.doubleClick()
                        cv2.putText(frame, 'Double Click', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                        click_times = []
                    else:
                        pyautogui.click()
                        cv2.putText(frame, 'Click', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

                    if freeze_cursor:
                        time.sleep(click_cooldown)
                    freeze_cursor = False

            prev_screen_x, prev_screen_y = screen_x, screen_y

            # scroll mode
            if sum(fingers) == 4 and fingers[0] == 0:  # All fingers except thumb are up
                scroll_mode = True
            else:
                scroll_mode = False
        
        
        #scroll actions
            if scroll_mode: 
                if index_tip.y < 0.4:
                    pyautogui.scroll(60)  # Scroll up
                    cv2.putText(frame, 'Scroll Up', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)   
                elif index_tip.y > 0.6:
                    pyautogui.scroll(-60)  # Scroll down
                    cv2.putText(frame, 'Scroll Down', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        
        
        #screenshot with ring finger
            if fingers[3] == 1 and time.time() - last_screenshot_time > screenshot_cooldown:
                pyautogui.screenshot(f'screenshot{int(time.time())}.png')
                last_screenshot_time = time.time()
                cv2.putText(frame, 'Screenshot Taken', (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    



    #fatigue detection
    if face_results.multi_face_landmarks:
       for face_landmarks in face_results.multi_face_landmarks:

            left_eye_top = face_landmarks.landmark[159]
            left_eye_bottom = face_landmarks.landmark[145]

            eye_dist = abs(left_eye_top.y - left_eye_bottom.y)

            if eye_dist < 0.01:
                eye_closed_frames += 1
            else:
                eye_closed_frames = 0

            if eye_closed_frames > 15:
                cv2.putText(frame, "DROWSY 😴", (10,120),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow('frame', frame)
    if cv2.waitKey(1) == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()   

In [2]:
model_path = "hand_landmarker.task"

In [3]:
base_options = python.BaseOptions(model_asset_path=model_path)

In [4]:
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2
)

In [5]:
landmarker = vision.HandLandmarker.create_from_options(options)

I0000 00:00:1774619493.808979  376279 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774619493.814488  376280 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774619493.818969  376280 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
detector = vision.HandLandmarker.create_from_options(options)

I0000 00:00:1774619497.193891  376403 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
W0000 00:00:1774619497.204519  376404 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774619497.214362  376404 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [7]:
mp_image = mp.Image.create_from_file('Circle-005.png')

In [8]:
# Perform hand landmarks detection on the provided single image.
# The hand landmarker must be created with the image mode.
hand_landmarker_result = landmarker.detect(mp_image)
print(hand_landmarker_result)
    

HandLandmarkerResult(handedness=[[Category(index=0, score=0.9890627264976501, display_name='Right', category_name='Right')]], hand_landmarks=[[NormalizedLandmark(x=0.397530734539032, y=0.6028016209602356, z=7.475540542145609e-07, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.5305988192558289, y=0.5402852892875671, z=-0.00709715997800231, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.6340951323509216, y=0.4461864233016968, z=-0.012937736697494984, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.7160654664039612, y=0.3950470983982086, z=-0.03154151514172554, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.7535146474838257, y=0.3408898413181305, z=-0.045752279460430145, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.499045193195343, y=0.31756314635276794, z=0.011289135552942753, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.589273989200592, y=0.23955008387565613, z=-0.0

W0000 00:00:1774619501.682482  376285 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
